In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["DEEPSEEK_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")

# Overview
Memory is a system that remembers information about previous interactions. For AI agents, memory is crucial because it lets them remember previous interactions, learn from feedback, and adapt to user preferences. As agents tackle more complex tasks with numerous user interactions, this capability becomes essential for both efficiency and user satisfaction.
Short term memory lets your application remember previous interactions within a single thread or conversation.

## Usage
To add short-term memory (thread-level persistence) to an agent, you need to specify a checkpointer when creating an agent.

In [10]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=50,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
)

agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Bob."}]},
    {"configurable": {"thread_id": "1"}},
)

{'messages': [HumanMessage(content='Hi! My name is Bob.', additional_kwargs={}, response_metadata={}, id='6d271091-db0c-4638-8215-673434e523ef'),
  AIMessage(content='Hi Bob! Nice to meet you. How can I help you today?', additional_kwargs={}, response_metadata={'id': 'msg_01BQfDCEkMLJXW8hrCoSmTXc', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 14, 'output_tokens': 18, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019cdbf9-cff9-70f3-ac58-bacd7e7eae51-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 18, 'total_tokens': 32, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, '

## Customizing agent memory
By default, agents use AgentState to manage short term memory, specifically the conversation history via a messages key.
You can extend AgentState to add additional fields. Custom state schemas are passed to create_agent using the state_schema parameter.

In [13]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):
    user_id: str
    prefrences: dict


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=50,
    timeout=30,
    max_retries=3,
)


agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    context_schema=CustomAgentState,
)

# Custom state can be passed in invoke
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Hello"}],
        "user_id": "user_123",
        "preferences": {"theme": "dark"},
    },
    {"configurable": {"thread_id": "1"}},
)

result["messages"][-1]

AIMessage(content='Hello! 👋 How can I help you today?', additional_kwargs={}, response_metadata={'id': 'msg_01KcthpoiuanQNqpiig56MqC', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 8, 'output_tokens': 16, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019cdbfc-ecb1-7002-9ae3-008de29542d2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 16, 'total_tokens': 24, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}})

# Common Patterns for Managing Conversation Memory

When **short-term memory** is enabled, long conversations may eventually **exceed the LLM’s context window**.  
To handle this, several common strategies are used to manage and optimize conversation history.

## Common Solutions

### 1. Trim Messages
Remove the **first or last N messages** from the conversation before sending the request to the LLM.

**Use case:**  
Helps keep only the most recent and relevant messages in the context.

---

### 2. Delete Messages
Permanently delete messages from the **LangGraph state**.

**Use case:**  
Useful when certain messages are no longer needed and should not be retained in memory.

---

### 3. Summarize Messages
Summarize earlier messages and replace them with a **compact summary**.

**Use case:**  
Maintains the conversation history while significantly reducing the number of tokens.

---

### 4. Custom Strategies
Implement custom approaches such as:

- Message filtering
- Priority-based message selection
- Topic-based conversation pruning
- Hybrid summarization + trimming

**Use case:**  
Allows developers to design memory strategies tailored to their application.

---

## Key Benefit

These techniques allow an **agent to maintain conversation context** while ensuring the total token count **does not exceed the LLM's context window**.


## 1: Trim messages
Most LLMs have a maximum supported context window (denominated in tokens).
One way to decide when to truncate messages is to count the tokens in the message history and truncate whenever it approaches that limit. If you’re using LangChain, you can use the trim messages utility and specify the number of tokens to keep from the list, as well as the strategy (e.g., keep the last max_tokens) to use for handling the boundary.
To trim message history in an agent, use the @before_model middleware decorator:

In [15]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any
from langchain_anthropic import ChatAnthropic


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *new_messages]}


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=50,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()
"""
================================== Ai Message ==================================

Your name is Bob. You told me that earlier.
If you'd like me to call you a nickname or use a different name, just say the word.
"""

================================== Ai Message ==================================

Your name is Bob! You told me at the start of our conversation.


"\n================================== Ai Message ==================================\n\nYour name is Bob. You told me that earlier.\nIf you'd like me to call you a nickname or use a different name, just say the word.\n"

## Delete messages
You can delete messages from the graph state to manage the message history.
This is useful when you want to remove specific messages or clear the entire message history.
To delete messages from the graph state, you can use the RemoveMessage.
For RemoveMessage to work, you need to use a state key with add_messages reducer.
The default AgentState provides this.
To remove specific messages:

In [16]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from langchain_anthropic import ChatAnthropic


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove old messages to keep conversation manageable."""
    messages = state["messages"]
    if len(messages) > 2:
        # remove the earliest two messages
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:2]]}
    return None


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=50,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="Please be concise and to the point.",
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

for event in agent.stream(
    {"messages": [{"role": "user", "content": "hi! I'm bob"}]},
    config,
    stream_mode="values",
):
    print([(message.type, message.content) for message in event["messages"]])

for event in agent.stream(
    {"messages": [{"role": "user", "content": "what's my name?"}]},
    config,
    stream_mode="values",
):
    print([(message.type, message.content) for message in event["messages"]])

[('human', "hi! I'm bob")]
[('human', "hi! I'm bob"), ('ai', 'Hi Bob! Nice to meet you. How can I help you today?')]
[('human', "hi! I'm bob"), ('ai', 'Hi Bob! Nice to meet you. How can I help you today?'), ('human', "what's my name?")]
[('human', "hi! I'm bob"), ('ai', 'Hi Bob! Nice to meet you. How can I help you today?'), ('human', "what's my name?"), ('ai', 'Your name is Bob! You told me that at the start of our conversation.')]
[('human', "what's my name?"), ('ai', 'Your name is Bob! You told me that at the start of our conversation.')]


## Summarize messages
The problem with trimming or removing messages, as shown above, is that you may lose information from culling of the message queue. Because of this, some applications benefit from a more sophisticated approach of summarizing the message history using a chat model.
Summary
To summarize message history in an agent, use the built-in SummarizationMiddleware:

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_anthropic import ChatAnthropic

checkpointer = InMemorySaver()


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=50,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model, trigger=("tokens", 100), keep=("messages", 20)
        )
    ],
    checkpointer=checkpointer,
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()
"""
================================== Ai Message ==================================

Your name is Bob!
"""

================================== Ai Message ==================================

Your name is Bob! You told me at the beginning of our conversation.


'\n================================== Ai Message ==================================\n\nYour name is Bob!\n'

## Access memory
You can access and modify the short-term memory (state) of an agent in several ways:

## Tools
### Read short-term memory in a tool
Access short term memory (state) in a tool using the runtime parameter (typed as ToolRuntime).
The runtime parameter is hidden from the tool signature (so the model doesn’t see it), but the tool can access the state through it.

In [23]:
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langchain_anthropic import ChatAnthropic


class CustomState(AgentState):
    user_id: str


@tool
def get_user_info(runtime: ToolRuntime) -> str:
    """Look up user info."""
    user_id = runtime.state["user_id"]
    return "User is John Smith" if user_id == "user_123" else "Unknown user"


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=50,
    timeout=30,
    max_retries=3,
)


agent = create_agent(
    model=model,
    tools=[get_user_info],
    state_schema=CustomState,
)

result = agent.invoke({"messages": "look up user information", "user_id": "user_123"})
print(result["messages"][-1].content)
# > User is John Smith.

The user information has been retrieved. The user is **John Smith**.


### Write short-term memory from tools
To modify the agent’s short-term memory (state) during execution, you can return state updates directly from the tools.
This is useful for persisting intermediate results or making information accessible to subsequent tools or prompts.

In [26]:
from langchain.tools import tool, ToolRuntime
from langchain_core.runnables import RunnableConfig
from langchain.messages import ToolMessage
from langchain.agents import create_agent, AgentState
from langgraph.types import Command
from pydantic import BaseModel
from langchain_anthropic import ChatAnthropic


class CustomState(AgentState):
    user_name: str


class CustomContext(BaseModel):
    user_id: str


@tool
def update_user_info(
    runtime: ToolRuntime[CustomContext, CustomState],
) -> Command:
    """Look up and update user info."""
    user_id = runtime.context.user_id
    name = "John Smith" if user_id == "user_123" else "Unknown user"
    return Command(
        update={
            "user_name": name,
            # update the message history
            "messages": [
                ToolMessage(
                    "Successfully looked up user information",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


@tool
def greet(runtime: ToolRuntime[CustomContext, CustomState]) -> str | Command:
    """Use this to greet the user once you found their info."""
    user_name = runtime.state.get("user_name", None)
    if user_name is None:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        "Please call the 'update_user_info' tool it will get and update the user's name.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )
    return f"Hello {user_name}!"


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=50,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model,
    tools=[update_user_info, greet],
    state_schema=CustomState,
    context_schema=CustomContext,
)

agent.invoke(
    {"messages": [{"role": "user", "content": "greet the user"}]},
    context=CustomContext(user_id="user_123"),
)

{'messages': [HumanMessage(content='greet the user', additional_kwargs={}, response_metadata={}, id='d0432828-295a-4b81-8bce-c0486cd98ece'),
  AIMessage(content=[{'id': 'toolu_01NWWyVccimVhC6Q37TESoY2', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'update_user_info', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01QnwMwyjnBpBhkbxLNCTqAZ', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 594, 'output_tokens': 39, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019cdc30-37dd-7d03-9563-7abbb0bf6dc5-0', tool_calls=[{'name': 'update_user_info', 'args': {}, 'id': 'toolu_01NWWyVccimVhC6Q37TESoY2', 'type': '

### Prompt
Access short term memory (state) in middleware to create dynamic prompts based on conversation history or custom state fields.

In [1]:
from langchain.agents import create_agent
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_anthropic import ChatAnthropic


class CustomContext(TypedDict):
    user_name: str


def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is always sunny!"


@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context["user_name"]
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}."
    return system_prompt


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens=50,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[dynamic_system_prompt],
    context_schema=CustomContext,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    context=CustomContext(user_name="John Smith"),
)
for msg in result["messages"]:
    msg.pretty_print()

KeyboardInterrupt: 

# Before Model Middleware

In an agent workflow, you can access **short-term memory (state)** using the `@before_model` middleware.  
This middleware runs **before the model is called**, allowing you to **inspect or modify messages** before they are sent to the LLM.

Typical use cases include:

- Trimming long conversations
- Filtering messages
- Injecting system prompts
- Logging requests
- Applying custom memory strategies

---

## Why Use `before_model`?

Long conversations can exceed the **LLM context window**, so processing messages before sending them to the model helps control:

- token usage
- memory size
- conversation quality
- system prompts

---

## Workflow

The agent execution flow looks like this:



### Step Explanation

| Step | Description |
|-----|-------------|
| `__start__` | Agent execution begins |
| `before_model` | Middleware processes state/messages before the model call |
| `model` | LLM generates a response |
| `tools` | Agent may call tools if required |
| `__end__` | Execution finishes |

---

## Example Use Cases

### 1. Trim Conversation History
Remove older messages to stay within the context window.

### 2. Inject System Prompt
Add a system instruction before sending messages to the model.

### 3. Filter Sensitive Data
Remove private or irrelevant messages.

### 4. Logging
Track messages for monitoring or debugging.

---

## Key Benefit

Using `@before_model` allows you to **control what the LLM sees**, ensuring:

- efficient token usage
- cleaner prompts
- better model responses

In [2]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime
from typing import Any
from langchain_anthropic import ChatAnthropic


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *new_messages]}


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens=50,
    timeout=30,
    max_retries=3,
)
agent = create_agent(
    model=model, tools=[], middleware=[trim_messages], checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()
"""
================================== Ai Message ==================================

Your name is Bob. You told me that earlier.
If you'd like me to call you a nickname or use a different name, just say the word.
"""

================================== Ai Message ==================================

Your name is Bob! You told me at the start of our conversation.


"\n================================== Ai Message ==================================\n\nYour name is Bob. You told me that earlier.\nIf you'd like me to call you a nickname or use a different name, just say the word.\n"

# After Model Middleware

In an agent workflow, you can access **short-term memory (state)** using the `@after_model` middleware.  
This middleware runs **after the model generates a response**, allowing you to **inspect or modify the model output** before the agent proceeds to the next step (such as tool execution).

Typical use cases include:

- Post-processing model responses
- Logging model outputs
- Validating or filtering responses
- Updating conversation state
- Triggering additional actions based on model output

---

## Why Use `after_model`?

Once the LLM produces a response, you may want to:

- clean or format the output
- extract structured information
- store results in memory
- detect tool calls or follow-up actions

This helps improve **control, monitoring, and reliability** of the agent system.

---

## Workflow

The agent execution flow looks like this:

### Step Explanation

| Step | Description |
|-----|-------------|
| `__start__` | Agent execution begins |
| `model` | LLM generates a response |
| `after_model` | Middleware processes the model output |
| `tools` | Agent may execute tools based on model output |
| `__end__` | Execution finishes |

---

## Example Use Cases

### 1. Output Formatting
Clean or restructure the model's response before returning it to the user.

### 2. Response Validation
Ensure the model output follows a required format (JSON, schema, etc.).

### 3. Logging & Monitoring
Store the model output for analytics or debugging.

### 4. State Updates
Update conversation memory or counters based on the model’s response.

---

## Key Benefit

Using `@after_model` allows developers to **control what happens after the model responds**, enabling:

- better output handling
- improved reliability
- structured response processing
- enhanced observability in agent workflows

In [5]:
from langchain.messages import RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.runtime import Runtime
from langchain_anthropic import ChatAnthropic


@after_model
def validate_response(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove messages containing sensitive words."""
    STOP_WORDS = ["password", "secret"]
    last_message = state["messages"][-1]
    if any(word in last_message.content for word in STOP_WORDS):
        return {"messages": [RemoveMessage(id=last_message.id)]}
    return None


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens=50,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[validate_response],
    checkpointer=InMemorySaver(),
)